# 07 DQA Scene Learned Adaptive Policy 8h

This notebook keeps the 06 scene/class DQA schedule but replaces the hand-written
adaptive pseudoGT rule with the lightweight learned threshold policy trained from
DQA05 logs.

- server labeled data is limited to the cloudy/partly-cloudy server split
- clients are unlabeled scene domains: highway, city street, residential
- warmup is trained in this workspace instead of copied from an older run
- phase2 pseudo-label gates are predicted from previous-round client/class stats
- learned proposals are clipped and smoothed so they stay close to the stable 05 regime

Default runtime target is similar to 06 on the current 2x RTX 6000 Ada setup:
20 warmup epochs plus 36 federated rounds (`12 + 24`).


## 1. Paths

In [1]:
from __future__ import annotations

import json
import os
import re
import shutil
import socket
import subprocess
import sys
from collections import deque
from pathlib import Path
from typing import Optional

import pandas as pd


def find_repo_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    required = (
        "dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_learned_adaptive_policy.py",
        "dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py",
        "navigating_data_heterogeneity/setup_fedsto_scene_reproduction.py",
    )
    for base in (start, *start.parents):
        for candidate in (base, base / "Object_Detection"):
            if all((candidate / marker).exists() for marker in required):
                return candidate.resolve()
    raise FileNotFoundError("Could not locate /app/Object_Detection")


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_learned_adaptive_policy.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
POLICY_MODEL = DQA_ROOT / "threshold_policy_model" / "artifacts" / "dqa05_threshold_policy.joblib"

WORK_ROOT = DQA_ROOT / "efficientteacher_dqa07_scene_learned_adaptive_policy_8h"
STATS_ROOT = DQA_ROOT / "stats_dqa07_scene_learned_adaptive_policy_8h"
RUNNER_LOG = DQA_ROOT / "dqa07_scene_learned_adaptive_policy_8h_runner.out"
TRAIN_LOG = DQA_ROOT / "dqa07_scene_learned_adaptive_policy_8h_train.log"
PID_PATH = DQA_ROOT / "dqa07_scene_learned_adaptive_policy_8h_runner.pid"
THRESHOLD_LOG = STATS_ROOT / "learned_adaptive_policy_schedule.jsonl"

preferred_python = Path("/root/micromamba/envs/al_yolov8/bin/python")
PYTHON_BIN = preferred_python if preferred_python.exists() else Path(sys.executable)

print("repo_root:", REPO_ROOT)
print("workspace:", WORK_ROOT)
print("stats_root:", STATS_ROOT)
print("python:", PYTHON_BIN)
print("policy_model:", POLICY_MODEL)
print("threshold_log:", THRESHOLD_LOG)


repo_root: /app/Object_Detection
workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h
stats_root: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa07_scene_learned_adaptive_policy_8h
python: /root/micromamba/envs/al_yolov8/bin/python
policy_model: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/threshold_policy_model/artifacts/dqa05_threshold_policy.joblib
threshold_log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa07_scene_learned_adaptive_policy_8h/learned_adaptive_policy_schedule.jsonl


## 2. Experiment Settings

In [2]:
# 8h learned adaptive policy profile.  Same schedule as 06, but warmup is trained here.
WARMUP_EPOCHS = 20
PHASE1_ROUNDS = 12
PHASE2_ROUNDS = 24
DQA_START_PHASE = 2

BATCH_SIZE = 160
WORKERS = 8
REQUESTED_GPUS = 2
MIN_FREE_GIB = 8

# Learned policy profile selected by run_dqa_cwa_fedsto_scene_v2_learned_adaptive_policy.py.
SSOD_PROFILE = "learned_adaptive_policy"
CLIENT_LR0 = 3e-4
SERVER_LR0 = 1e-3

# The first two phase2 rounds stay on the 05-compatible fixed gate.  From round 3,
# the DQA05 policy proposes class-wise gates using previous-round pseudo stats.
ADAPT_START_ROUND = 3
POLICY_HORIZON_ROUNDS = PHASE2_ROUNDS
MAX_LOW = 0.46
MAX_HIGH = 0.87
MAX_NMS = 0.42
TEACHER_MIN = 0.30
RARE_COUNT = 250
RARE_MAX_LOW = 0.42
RARE_MAX_HIGH = 0.84
LOW_STEP_LIMIT = 0.02
HIGH_STEP_LIMIT = 0.035
NMS_STEP_LIMIT = 0.02

# Warmup is intentionally not seeded from the old 05/06 source checkpoint.
FORCE_WARMUP = False
APPEND_TRAIN_LOG = False
RUN_TRAINING = True
RUN_IN_BACKGROUND = False
STREAM_TRAIN_OUTPUT = True

try:
    import torch

    AVAILABLE_CUDA_GPUS = torch.cuda.device_count()
except Exception as exc:
    AVAILABLE_CUDA_GPUS = 0
    print("Could not inspect CUDA devices:", exc)

GPUS = min(REQUESTED_GPUS, AVAILABLE_CUDA_GPUS) if AVAILABLE_CUDA_GPUS else 1
if GPUS != REQUESTED_GPUS:
    print(f"Requested {REQUESTED_GPUS} GPU(s), visible={AVAILABLE_CUDA_GPUS}; using GPUS={GPUS}")


def find_free_port(preferred: int) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind(("127.0.0.1", preferred))
            return preferred
        except OSError:
            pass
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


MASTER_PORT = find_free_port(29555)

os.environ["DQA07_SSOD_PROFILE"] = SSOD_PROFILE
os.environ["DQA07_POLICY_MODEL"] = str(POLICY_MODEL)
os.environ["DQA07_POLICY_HORIZON_ROUNDS"] = str(POLICY_HORIZON_ROUNDS)
os.environ["DQA07_CLIENT_LR0"] = str(CLIENT_LR0)
os.environ["DQA07_SERVER_LR0"] = str(SERVER_LR0)
os.environ["DQA07_ADAPT_START_ROUND"] = str(ADAPT_START_ROUND)
os.environ["DQA07_MAX_LOW"] = str(MAX_LOW)
os.environ["DQA07_MAX_HIGH"] = str(MAX_HIGH)
os.environ["DQA07_MAX_NMS"] = str(MAX_NMS)
os.environ["DQA07_TEACHER_MIN"] = str(TEACHER_MIN)
os.environ["DQA07_RARE_COUNT"] = str(RARE_COUNT)
os.environ["DQA07_RARE_MAX_LOW"] = str(RARE_MAX_LOW)
os.environ["DQA07_RARE_MAX_HIGH"] = str(RARE_MAX_HIGH)
os.environ["DQA07_LOW_STEP_LIMIT"] = str(LOW_STEP_LIMIT)
os.environ["DQA07_HIGH_STEP_LIMIT"] = str(HIGH_STEP_LIMIT)
os.environ["DQA07_NMS_STEP_LIMIT"] = str(NMS_STEP_LIMIT)
os.environ["DQA07_THRESHOLD_LOG"] = str(THRESHOLD_LOG)

{
    "phase1_rounds": PHASE1_ROUNDS,
    "phase2_rounds": PHASE2_ROUNDS,
    "dqa_start_phase": DQA_START_PHASE,
    "ssod_profile": SSOD_PROFILE,
    "policy_model_exists": POLICY_MODEL.exists(),
    "client_lr0": CLIENT_LR0,
    "server_lr0": SERVER_LR0,
    "adapt_start_round": ADAPT_START_ROUND,
    "max_low": MAX_LOW,
    "max_high": MAX_HIGH,
    "max_nms": MAX_NMS,
    "threshold_log": str(THRESHOLD_LOG),
    "batch_size": BATCH_SIZE,
    "gpus": GPUS,
    "master_port": MASTER_PORT,
    "workspace": str(WORK_ROOT),
}


{'phase1_rounds': 12,
 'phase2_rounds': 24,
 'dqa_start_phase': 2,
 'ssod_profile': 'learned_adaptive_policy',
 'policy_model_exists': True,
 'client_lr0': 0.0003,
 'server_lr0': 0.001,
 'adapt_start_round': 3,
 'max_low': 0.46,
 'max_high': 0.87,
 'max_nms': 0.42,
 'threshold_log': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa07_scene_learned_adaptive_policy_8h/learned_adaptive_policy_schedule.jsonl',
 'batch_size': 160,
 'gpus': 2,
 'master_port': 29555,
 'workspace': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h'}

## 3. Build Lists and Warmup Policy

The scene setup creates highway/citystreet/residential clients and scene-wise validation
lists.  The 8h adaptive profile reuses the existing scene warmup so the budget is spent on DQA and pseudoGT gating.

In [3]:
subprocess.run(
    [
        str(PYTHON_BIN),
        str(RUN_SCRIPT),
        "--setup-only",
        "--workspace-root",
        str(WORK_ROOT),
        "--stats-root",
        str(STATS_ROOT),
    ],
    cwd=REPO_ROOT,
    check=True,
    env=os.environ.copy(),
)

if not POLICY_MODEL.exists():
    raise FileNotFoundError(f"Learned policy model is missing: {POLICY_MODEL}")

warmup_dst = WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"
if warmup_dst.exists() and not FORCE_WARMUP:
    print("Warmup already present and will be reused unless FORCE_WARMUP=True:", warmup_dst)
elif FORCE_WARMUP:
    print("FORCE_WARMUP=True; runner will retrain warmup.")
else:
    print("No warmup seed will be copied; the runner will train warmup from scratch.")

manifest = json.loads((WORK_ROOT / "manifest.json").read_text(encoding="utf-8"))
server = manifest["server"]
clients = manifest["clients"]
eval_splits = manifest["paper_evaluation"]["splits"]

display(pd.DataFrame([server]))
display(pd.DataFrame(clients))
display(pd.DataFrame(eval_splits)[["name", "raw_scene", "images", "boxes"]])


{
  "manifest": {
    "paper": "Navigating Data Heterogeneity in Federated Learning: A Semi-Supervised Federated Object Detection",
    "variant": "scene clients: highway, city street, residential",
    "official_ssfod_repo": "https://github.com/Kthyeon/ssfod",
    "official_ssfod_sha": "c411d22607c07a67ef632834eb15575f1df5c5f2",
    "efficientteacher_repo": "https://github.com/AlibabaResearch/efficientteacher",
    "efficientteacher_sha": "c411d22607c07a67ef632834eb15575f1df5c5f2",
    "server": {
      "weather": "cloudy represented by BDD100K Kaggle weather='partly cloudy'",
      "train_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h/data_lists/server_cloudy_train.txt",
      "val_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h/data_lists/paper_eval_scene_total_val.txt",
      "source_val_list": "/app/Object_Detection/dy

,weather,train_list,val_list,source_val_list,train_images,val_images,source_val_images,validation_target
0,cloudy represented by BDD100K Kaggle weather='...,/app/Object_Detection/dynamic_quality_aware_cl...,/app/Object_Detection/dynamic_quality_aware_cl...,/app/Object_Detection/dynamic_quality_aware_cl...,4881,9864,738,scene_total


,id,weather,scene,list,images
0,0,highway,highway,/app/Object_Detection/dynamic_quality_aware_cl...,5000
1,1,citystreet,city street,/app/Object_Detection/dynamic_quality_aware_cl...,5000
2,2,residential,residential,/app/Object_Detection/dynamic_quality_aware_cl...,5000


,name,raw_scene,images,boxes
0,highway,highway,2499,36377
1,citystreet,city street,6112,127178
2,residential,residential,1253,20855


## 4. Dry Run

In [4]:
dry_cmd = [
    str(PYTHON_BIN),
    str(RUN_SCRIPT),
    "--dry-run",
    "--workspace-root",
    str(WORK_ROOT),
    "--stats-root",
    str(STATS_ROOT),
    "--warmup-epochs",
    str(WARMUP_EPOCHS),
    "--phase1-rounds",
    str(PHASE1_ROUNDS),
    "--phase2-rounds",
    str(PHASE2_ROUNDS),
    "--dqa-start-phase",
    str(DQA_START_PHASE),
    "--batch-size",
    str(BATCH_SIZE),
    "--workers",
    str(WORKERS),
    "--gpus",
    str(GPUS),
    "--master-port",
    str(MASTER_PORT),
    "--min-free-gib",
    str(MIN_FREE_GIB),
    "--classwise-blend",
    "0.35",
    "--server-anchor",
    "1.25",
    "--localize-bn",
    "--enable-dqa-guard",
    "--dqa-drop-ratio-threshold",
    "0.15",
    "--dqa-spike-ratio-threshold",
    "3.0",
]
if FORCE_WARMUP:
    dry_cmd.append("--force-warmup")

subprocess.run(dry_cmd, cwd=REPO_ROOT, check=True, env=os.environ.copy())

DQA07 profile: learned_adaptive_policy
DQA07 policy model: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/threshold_policy_model/artifacts/dqa05_threshold_policy.joblib
{
  "manifest": {
    "paper": "Navigating Data Heterogeneity in Federated Learning: A Semi-Supervised Federated Object Detection",
    "variant": "scene clients: highway, city street, residential",
    "official_ssfod_repo": "https://github.com/Kthyeon/ssfod",
    "official_ssfod_sha": "c411d22607c07a67ef632834eb15575f1df5c5f2",
    "efficientteacher_repo": "https://github.com/AlibabaResearch/efficientteacher",
    "efficientteacher_sha": "c411d22607c07a67ef632834eb15575f1df5c5f2",
    "server": {
      "weather": "cloudy represented by BDD100K Kaggle weather='partly cloudy'",
      "train_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h/data_lists/server_cloudy_train.txt",
      "val_list": "/app/Object_Detection/dynami

CompletedProcess(args=['/root/micromamba/envs/al_yolov8/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_learned_adaptive_policy.py', '--dry-run', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h', '--stats-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa07_scene_learned_adaptive_policy_8h', '--warmup-epochs', '20', '--phase1-rounds', '12', '--phase2-rounds', '24', '--dqa-start-phase', '2', '--batch-size', '160', '--workers', '8', '--gpus', '2', '--master-port', '29555', '--min-free-gib', '8', '--classwise-blend', '0.35', '--server-anchor', '1.25', '--localize-bn', '--enable-dqa-guard', '--dqa-drop-ratio-threshold', '0.15', '--dqa-spike-ratio-threshold', '3.0'], returncode=0)

## 5. Start or Resume Training

In [5]:
def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text(encoding="utf-8").strip())
    except ValueError:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "missing"
    result = subprocess.run(["ps", "-o", "stat=", "-p", str(pid)], capture_output=True, text=True)
    state = result.stdout.strip()
    if result.returncode != 0 or not state:
        return "missing"
    if "Z" in state:
        return "zombie"
    return state


train_cmd = [
    str(PYTHON_BIN),
    "-u",
    str(RUN_SCRIPT),
    "--workspace-root",
    str(WORK_ROOT),
    "--stats-root",
    str(STATS_ROOT),
    "--warmup-epochs",
    str(WARMUP_EPOCHS),
    "--phase1-rounds",
    str(PHASE1_ROUNDS),
    "--phase2-rounds",
    str(PHASE2_ROUNDS),
    "--dqa-start-phase",
    str(DQA_START_PHASE),
    "--batch-size",
    str(BATCH_SIZE),
    "--workers",
    str(WORKERS),
    "--gpus",
    str(GPUS),
    "--master-port",
    str(MASTER_PORT),
    "--min-free-gib",
    str(MIN_FREE_GIB),
    "--log-file",
    str(TRAIN_LOG),
    "--classwise-blend",
    "0.35",
    "--server-anchor",
    "1.25",
    "--localize-bn",
    "--enable-dqa-guard",
    "--dqa-drop-ratio-threshold",
    "0.15",
    "--dqa-spike-ratio-threshold",
    "3.0",
]
if FORCE_WARMUP:
    train_cmd.append("--force-warmup")
if APPEND_TRAIN_LOG:
    train_cmd.append("--append-train-log")
if STREAM_TRAIN_OUTPUT:
    train_cmd.append("--stream-train-output")

current_pid = read_pid(PID_PATH)
state = pid_state(current_pid)
print("existing pid:", current_pid, state)
print(" ".join(train_cmd))

if RUN_TRAINING and state not in {"missing", "zombie"}:
    print("Training already appears to be running.")
elif RUN_TRAINING and RUN_IN_BACKGROUND:
    env = os.environ.copy()
    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_mode = "ab" if APPEND_TRAIN_LOG else "wb"
    with RUNNER_LOG.open(log_mode) as out:
        process = subprocess.Popen(
            train_cmd,
            cwd=REPO_ROOT,
            stdout=out,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )
    PID_PATH.write_text(str(process.pid), encoding="utf-8")
    print("Started PID:", process.pid)
    print("Runner log:", RUNNER_LOG)
    print("Train log:", TRAIN_LOG)
elif RUN_TRAINING:
    env = os.environ.copy()
    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_mode = "a" if APPEND_TRAIN_LOG else "w"
    print("Running in foreground; progress will stream in this cell.")
    print("Runner log:", RUNNER_LOG)
    print("Train log:", TRAIN_LOG)
    with RUNNER_LOG.open(log_mode, encoding="utf-8", buffering=1) as runner_log:
        process = subprocess.Popen(
            train_cmd,
            cwd=REPO_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        PID_PATH.write_text(str(process.pid), encoding="utf-8")
        print("Started PID:", process.pid)
        runner_log.write(f"Started PID: {process.pid}\n")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            runner_log.write(line)
        return_code = process.wait()
    if PID_PATH.exists() and PID_PATH.read_text(encoding="utf-8").strip() == str(process.pid):
        PID_PATH.unlink()
    if return_code != 0:
        raise RuntimeError(f"Training failed with exit code {return_code}. See {RUNNER_LOG} and {TRAIN_LOG}.")
    print("Training completed.")
else:
    print("RUN_TRAINING=False, command was not launched.")

existing pid: None missing
/root/micromamba/envs/al_yolov8/bin/python -u /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_learned_adaptive_policy.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h --stats-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa07_scene_learned_adaptive_policy_8h --warmup-epochs 20 --phase1-rounds 12 --phase2-rounds 24 --dqa-start-phase 2 --batch-size 160 --workers 8 --gpus 2 --master-port 29555 --min-free-gib 8 --log-file /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/dqa07_scene_learned_adaptive_policy_8h_train.log --classwise-blend 0.35 --server-anchor 1.25 --localize-bn --enable-dqa-guard --dqa-drop-ratio-threshold 0.15 --dqa-spike-ratio-threshold 3.0 --stream-train-output
Running in foreground; progress will stream in this cell.
Runner log: /app/Object_Detection/dynamic_qu

## 6. Status

In [6]:
def tail_lines(path: Path, lines: int = 30) -> list[str]:
    if not path.exists():
        return []
    try:
        result = subprocess.run(["tail", "-n", str(lines), str(path)], capture_output=True, text=True, check=True)
        return result.stdout.splitlines()
    except Exception:
        with path.open(encoding="utf-8", errors="replace") as f:
            return [line.rstrip("\n") for line in deque(f, maxlen=lines)]


history_path = WORK_ROOT / "history.json"
history = json.loads(history_path.read_text(encoding="utf-8")) if history_path.exists() else []
pid = read_pid(PID_PATH)

completed_phase1 = sum(1 for row in history if int(row.get("phase", 0)) == 1)
completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
latest_global = Path(history[-1]["global"]) if history else WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"

display(
    pd.DataFrame(
        [
            {
                "pid": pid,
                "pid_state": pid_state(pid),
                "completed_phase1": f"{completed_phase1}/{PHASE1_ROUNDS}",
                "completed_phase2": f"{completed_phase2}/{PHASE2_ROUNDS}",
                "completed_total": f"{len(history)}/{PHASE1_ROUNDS + PHASE2_ROUNDS}",
                "latest_global": str(latest_global),
                "free_gib": round(shutil.disk_usage(WORK_ROOT).free / 1024**3, 2),
            }
        ]
    )
)

print("Runner log tail:")
for line in tail_lines(RUNNER_LOG, 35):
    print(line)
print("\nTrain log tail:")
for line in tail_lines(TRAIN_LOG, 35):
    print(line)

,pid,pid_state,completed_phase1,completed_phase2,completed_total,latest_global,free_gib
0,None,missing,12/12,24/24,36/36,/app/Object_Detection/dynamic_quality_aware_cl...,126.24


Runner log tail:
               Class     Images     Labels          P          R     mAP@.5 mAP@.5:.95:  80%|████████  | 4/5 [00:04<00:00,  1.07it/s]
               Class     Images     Labels          P          R     mAP@.5 mAP@.5:.95: 100%|██████████| 5/5 [00:06<00:00,  1.23s/it]
               Class     Images     Labels          P          R     mAP@.5 mAP@.5:.95: 100%|██████████| 5/5 [00:06<00:00,  1.32s/it]
                 all        738      14937      0.539      0.373      0.363      0.199
Optimizer stripped from /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h/runs/dqa_phase2_round024_server/weights/last.pt, 93.0MB
Optimizer stripped from /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h/runs/dqa_phase2_round024_server/weights/best.pt, 93.0MB

Validating /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficienttea

## 7. Scene/Class Evaluation

After the run finishes, set `RUN_EVAL=True`.  This evaluates warmup, final phase 1,
and final phase 2 on highway/citystreet/residential/total, with per-class AP rows.

In [7]:
RUN_EVAL = False
EVAL_SPLITS = "highway,citystreet,residential,total"
EVAL_BATCH_SIZE = 16
EVAL_DEVICE = ""

history = json.loads((WORK_ROOT / "history.json").read_text(encoding="utf-8")) if (WORK_ROOT / "history.json").exists() else []
checkpoints: list[tuple[str, Path]] = []
warmup = WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"
if warmup.exists():
    checkpoints.append(("warmup", warmup))
phase1 = [row for row in history if int(row.get("phase", 0)) == 1]
phase2 = [row for row in history if int(row.get("phase", 0)) == 2]
if phase1:
    checkpoints.append((f"phase1_round{int(phase1[-1]['round']):03d}", Path(phase1[-1]["global"])))
if phase2:
    checkpoints.append((f"phase2_round{int(phase2[-1]['round']):03d}", Path(phase2[-1]["global"])))

eval_cmd = [
    str(PYTHON_BIN),
    str(EVAL_SCRIPT),
    "--workspace",
    str(WORK_ROOT),
    "--splits",
    EVAL_SPLITS,
    "--batch-size",
    str(EVAL_BATCH_SIZE),
    "--no-plots",
    "--verbose",
]
if EVAL_DEVICE:
    eval_cmd.extend(["--device", EVAL_DEVICE])
for label, path in checkpoints:
    eval_cmd.extend(["--checkpoint", f"{label}={path}"])

print("checkpoints:", checkpoints)
print(" ".join(eval_cmd))
if RUN_EVAL and checkpoints:
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)
elif RUN_EVAL:
    print("No checkpoints found to evaluate.")
else:
    print("RUN_EVAL=False; set True after training finishes.")

checkpoints: [('warmup', PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h/global_checkpoints/round000_warmup.pt')), ('phase1_round012', PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h/global_checkpoints/phase1_round012_global.pt')), ('phase2_round024', PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h/global_checkpoints/phase2_round024_global.pt'))]
/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py --workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h --splits highway,citystreet,residential,total --batch-size 16 --no-plots --verbose --checkpoint warmup=/app/Object_Detection/d

## 8. Read Evaluation Tables

In [8]:
summary_csv = WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"
classwise_csv = WORK_ROOT / "validation_reports" / "paper_protocol_classwise_summary.csv"

if summary_csv.exists():
    summary = pd.read_csv(summary_csv)
    display(summary.sort_values(["checkpoint_label", "split"]))
else:
    print("No summary yet:", summary_csv)

if classwise_csv.exists():
    classwise = pd.read_csv(classwise_csv)
    display(classwise.sort_values(["split", "class", "map50_95"], ascending=[True, True, False]).head(80))
    pivot = classwise.pivot_table(
        index=["split", "class"],
        columns="checkpoint_label",
        values="map50_95",
        aggfunc="first",
    )
    display(pivot)
else:
    print("No classwise summary yet:", classwise_csv)

No summary yet: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h/validation_reports/paper_protocol_eval_summary.csv
No classwise summary yet: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa07_scene_learned_adaptive_policy_8h/validation_reports/paper_protocol_classwise_summary.csv


## 9. DQA Stats Snapshot

In [9]:
state_path = WORK_ROOT / "dqa_cwa_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    guard = state.get("round_guard", {})
    display(pd.DataFrame(guard.get("history", [])).tail(20))
    alpha = state.get("alpha", {})
    if alpha:
        latest_key = sorted(alpha)[-1]
        alpha_df = pd.DataFrame(alpha[latest_key]).T
        alpha_df.columns = latest_key.split("|")
        alpha_df.insert(0, "class", manifest["classes"])
        display(alpha_df)
else:
    print("No DQA state yet:", state_path)

stats_files = sorted(STATS_ROOT.glob("phase*_round*.json"))
print("stats files:", len(stats_files), "root:", STATS_ROOT)

,active_classes,mean_quality,phase,reason,round,total_count,used_dqa
4,8,0.766408,2,,5,403769.0,True
5,8,0.765748,2,,6,410823.0,True
6,8,0.766973,2,,7,412414.0,True
7,8,0.765736,2,,8,419354.0,True
8,8,0.764751,2,,9,425434.0,True
9,8,0.763410,2,,10,432093.0,True
10,8,0.763205,2,,11,435356.0,True
11,8,0.763091,2,,12,440321.0,True
12,8,0.765011,2,,13,440075.0,True
13,8,0.763089,2,,14,445433.0,True


,class,client:0,client:1,client:2,server
0,person,0.167252,0.196761,0.185987,0.450000
1,rider,0.012502,0.452232,0.012502,0.522765
2,car,0.179276,0.184113,0.186611,0.450000
3,bus,0.180843,0.193247,0.175911,0.450000
4,truck,0.185211,0.185407,0.179381,0.450000
5,bike,0.146559,0.203750,0.199691,0.450000
6,motor,0.250000,0.250000,0.250000,0.250000
7,traffic light,0.176879,0.191848,0.181273,0.450000
8,traffic sign,0.191229,0.181594,0.177177,0.450000
9,train,0.250000,0.250000,0.250000,0.250000


stats files: 96 root: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa07_scene_learned_adaptive_policy_8h


## 10. Learned Adaptive Policy Schedule

In [10]:
if THRESHOLD_LOG.exists():
    records = [json.loads(line) for line in THRESHOLD_LOG.read_text(encoding="utf-8").splitlines() if line.strip()]
    gate_df = pd.DataFrame(records)
    display(gate_df.tail(30))
    if not gate_df.empty:
        compact = gate_df[["phase", "round", "client_id", "enabled", "reason", "nms_conf_thres", "teacher_loss_weight", "source_stats"]].copy()
        compact["low_min"] = gate_df["ignore_thres_low"].map(lambda xs: min(xs) if isinstance(xs, list) else None)
        compact["low_max"] = gate_df["ignore_thres_low"].map(lambda xs: max(xs) if isinstance(xs, list) else None)
        compact["high_min"] = gate_df["ignore_thres_high"].map(lambda xs: min(xs) if isinstance(xs, list) else None)
        compact["high_max"] = gate_df["ignore_thres_high"].map(lambda xs: max(xs) if isinstance(xs, list) else None)
        display(compact.tail(40))
else:
    print("No learned adaptive policy log yet:", THRESHOLD_LOG)


,run_name,policy_model,enabled,reason,phase,round,client_id,nms_conf_thres,ignore_thres_low,ignore_thres_high,teacher_loss_weight,box_loss_weight,obj_loss_weight,cls_loss_weight,source_stats
42,dqa_phase2_round015_client0_highway,/app/Object_Detection/dynamic_quality_aware_cl...,True,learned-policy-previous-client-stats+smoothed,2,15,0,0.3627,"[0.3941, 0.4009, 0.3827, 0.374, 0.3784, 0.402,...","[0.8147, 0.8371, 0.7958, 0.7845, 0.7906, 0.837...",0.3307,0.01884,0.3346,0.0904,/app/Object_Detection/dynamic_quality_aware_cl...
43,dqa_phase2_round015_client1_citystreet,/app/Object_Detection/dynamic_quality_aware_cl...,True,learned-policy-previous-client-stats+smoothed,2,15,1,0.3600,"[0.396, 0.4044, 0.38, 0.3741, 0.3814, 0.4032, ...","[0.8179, 0.84, 0.7911, 0.7846, 0.7955, 0.8339,...",0.3300,0.01880,0.3340,0.0900,/app/Object_Detection/dynamic_quality_aware_cl...
44,dqa_phase2_round015_client2_residential,/app/Object_Detection/dynamic_quality_aware_cl...,True,learned-policy-previous-client-stats+smoothed,2,15,2,0.3599,"[0.394, 0.4012, 0.3799, 0.3746, 0.3758, 0.4031...","[0.8146, 0.8372, 0.791, 0.7854, 0.7867, 0.834,...",0.3305,0.01883,0.3344,0.0902,/app/Object_Detection/dynamic_quality_aware_cl...
45,dqa_phase2_round016_client0_highway,/app/Object_Detection/dynamic_quality_aware_cl...,True,learned-policy-previous-client-stats+smoothed,2,16,0,0.3716,"[0.4047, 0.4111, 0.3916, 0.3831, 0.3855, 0.412...","[0.8309, 0.84, 0.8093, 0.7988, 0.8016, 0.84, 0...",0.3263,0.01858,0.3310,0.0882,/app/Object_Detection/dynamic_quality_aware_cl...
46,dqa_phase2_round016_client1_citystreet,/app/Object_Detection/dynamic_quality_aware_cl...,True,learned-policy-previous-client-stats+smoothed,2,16,1,0.3688,"[0.4065, 0.4124, 0.3888, 0.3832, 0.3916, 0.414...","[0.8341, 0.84, 0.8045, 0.7989, 0.8117, 0.8511,...",0.3255,0.01853,0.3304,0.0877,/app/Object_Detection/dynamic_quality_aware_cl...
47,dqa_phase2_round016_client2_residential,/app/Object_Detection/dynamic_quality_aware_cl...,True,learned-policy-previous-client-stats+smoothed,2,16,2,0.3682,"[0.4041, 0.4113, 0.3882, 0.3833, 0.387, 0.4144...","[0.8301, 0.84, 0.8038, 0.7992, 0.8041, 0.8516,...",0.3260,0.01856,0.3308,0.0880,/app/Object_Detection/dynamic_quality_aware_cl...
48,dqa_phase2_round017_client0_highway,/app/Object_Detection/dynamic_quality_aware_cl...,True,learned-policy-previous-client-stats+smoothed,2,17,0,0.3732,"[0.4067, 0.416, 0.3932, 0.3863, 0.3892, 0.4171...","[0.8338, 0.84, 0.8113, 0.8037, 0.8074, 0.84, 0...",0.3247,0.01848,0.3298,0.0874,/app/Object_Detection/dynamic_quality_aware_cl...
49,dqa_phase2_round017_client1_citystreet,/app/Object_Detection/dynamic_quality_aware_cl...,True,learned-policy-previous-client-stats+smoothed,2,17,1,0.3723,"[0.4075, 0.42, 0.3923, 0.3865, 0.3955, 0.4172,...","[0.8352, 0.84, 0.8101, 0.804, 0.8176, 0.8554, ...",0.3238,0.01843,0.3291,0.0869,/app/Object_Detection/dynamic_quality_aware_cl...
50,dqa_phase2_round017_client2_residential,/app/Object_Detection/dynamic_quality_aware_cl...,True,learned-policy-previous-client-stats+smoothed,2,17,2,0.3718,"[0.4068, 0.4163, 0.3918, 0.3866, 0.3894, 0.417...","[0.834, 0.84, 0.8096, 0.8042, 0.8078, 0.8565, ...",0.3244,0.01846,0.3295,0.0872,/app/Object_Detection/dynamic_quality_aware_cl...
51,dqa_phase2_round018_client0_highway,/app/Object_Detection/dynamic_quality_aware_cl...,True,learned-policy-previous-client-stats+smoothed,2,18,0,0.3751,"[0.4076, 0.4192, 0.3951, 0.3886, 0.3908, 0.42,...","[0.835, 0.84, 0.814, 0.8073, 0.8099, 0.84, 0.8...",0.3236,0.01842,0.3289,0.0868,/app/Object_Detection/dynamic_quality_aware_cl...


,phase,round,client_id,enabled,reason,nms_conf_thres,teacher_loss_weight,source_stats,low_min,low_max,high_min,high_max
32,2,11,2,True,learned-policy-previous-client-stats+smoothed,0.3500,0.3359,/app/Object_Detection/dynamic_quality_aware_cl...,0.3624,0.3937,0.7659,0.8196
33,2,12,0,True,learned-policy-previous-client-stats+smoothed,0.3527,0.3343,/app/Object_Detection/dynamic_quality_aware_cl...,0.3653,0.3973,0.7708,0.8285
34,2,12,1,True,learned-policy-previous-client-stats+smoothed,0.3501,0.3335,/app/Object_Detection/dynamic_quality_aware_cl...,0.3652,0.3977,0.7708,0.8299
35,2,12,2,True,learned-policy-previous-client-stats+smoothed,0.3501,0.3340,/app/Object_Detection/dynamic_quality_aware_cl...,0.3654,0.3972,0.7709,0.8281
36,2,13,0,True,learned-policy-previous-client-stats+smoothed,0.3558,0.3330,/app/Object_Detection/dynamic_quality_aware_cl...,0.3681,0.3991,0.7754,0.8335
37,2,13,1,True,learned-policy-previous-client-stats+smoothed,0.3538,0.3322,/app/Object_Detection/dynamic_quality_aware_cl...,0.3682,0.3997,0.7755,0.8335
38,2,13,2,True,learned-policy-previous-client-stats+smoothed,0.3534,0.3327,/app/Object_Detection/dynamic_quality_aware_cl...,0.3680,0.3999,0.7752,0.8332
39,2,14,0,True,learned-policy-previous-client-stats+smoothed,0.3560,0.3328,/app/Object_Detection/dynamic_quality_aware_cl...,0.3690,0.3998,0.7766,0.8338
40,2,14,1,True,learned-policy-previous-client-stats+smoothed,0.3538,0.3319,/app/Object_Detection/dynamic_quality_aware_cl...,0.3689,0.4005,0.7765,0.8348
41,2,14,2,True,learned-policy-previous-client-stats+smoothed,0.3537,0.3323,/app/Object_Detection/dynamic_quality_aware_cl...,0.3691,0.4000,0.7769,0.8336
